# Fase 4 · Monitorización, alertas y estabilidad  (RA4 · UD4)

**Reto:** la plataforma pasa a «producción». El centro de control del puerto necesita saber **en todo momento** si
el clúster está sano, recibir **alertas** antes de que un fallo afecte al negocio y un **informe de estabilidad**
con datos objetivos.

Arranque (PowerShell): `.\bda.ps1 iniciar monitor` → Prometheus http://localhost:9090 · Grafana http://localhost:3000

| Parte | Qué hacéis | CE |
|---|---|---|
| A | Comprobar qué se monitoriza y ampliar el exportador HDFS | 4.a, 4.b |
| B | Generar carga y explorar métricas con PromQL | 4.b |
| C | Cuadro de mando en Grafana | 4.b |
| D | Alertas propias y medición del tiempo de detección | 4.c |
| E | Rendimiento de las herramientas de monitorización | 4.d |
| F | Fiabilidad: ¿dicen la verdad las métricas? | 4.e |
| G | Prueba de estabilidad con caos controlado | 4.f |

In [ ]:
from pyspark.sql import functions as F
from puerto.sesion import crear_spark, cronometro, sello
PAREJA = "00"   # ✏️ vuestro número de pareja: su salida debe verse en las capturas de la memoria
sello(PAREJA)
from puerto import hdfs_web as h, metricas as m
import pandas as pd, matplotlib.pyplot as plt, requests, time, os, re
os.makedirs("/workspace/entregas/F4", exist_ok=True)
pd.DataFrame(m.objetivos())

## A · ¿Qué estamos vigilando?
Cada fila anterior es un **objetivo** (*target*) que Prometheus consulta cada 15 s. `spark-driver` y
`spark-executors` solo están *up* mientras hay una SparkSession abierta.

Abrid en el navegador http://localhost:9870/jmx (JSON interno del NameNode) y comparadlo con lo que publica el
exportador: `docker compose exec jupyter curl -s http://hdfs-exporter:9101/metrics`

**✏️ TAREA A1.** Completad las dos TAREAS de `docker/exporter/hdfs_exporter.py` (≥ 3 métricas nuevas y una serie por
DataNode). Después: `docker compose --profile monitor restart hdfs-exporter` y comprobadlo en la celda siguiente.

In [ ]:
texto = requests.get("http://hdfs-exporter:9101/metrics", timeout=10).text
print("\n".join(l for l in texto.splitlines() if not l.startswith("#")))

## B · Carga de trabajo y PromQL
Lanzamos una carga repetitiva sobre el clúster (en esta misma sesión) y miramos cómo se refleja en las métricas.
Mientras se ejecuta, abrid Grafana y http://localhost:9090/graph.

In [ ]:
spark = crear_spark("F4-carga", cores_max=4)

def consulta_tipo():
    '''Consulta representativa del uso real: posiciones del último día disponible por estado de navegación.'''
    df = spark.read.parquet("/puerto/bronze/ais")
    ultimo = df.agg(F.max("fecha")).collect()[0][0]
    return df.filter(F.col("fecha") == ultimo).groupBy("estado_nav").count().collect()

def generar_carga(minutos=5):
    fin, n = time.time() + minutos * 60, 0
    while time.time() < fin:
        (spark.read.parquet("/puerto/bronze/ais")
              .withColumn("k", F.explode(F.sequence(F.lit(1), F.lit(3))))
              .groupBy("mmsi", "k").agg(F.avg("sog"), F.max("ts")).count())
        n += 1
    print("iteraciones de carga:", n)

generar_carga(5)

In [ ]:
consultas = {
    "CPU del nodo (%)": '100 * (1 - avg(rate(node_cpu_seconds_total{mode="idle"}[1m])))',
    "Memoria disponible (GB)": "node_memory_MemAvailable_bytes / 1e9",
    "Cores usados en workers": "sum(metrics_worker_coresUsed_Value)",
    "Tareas activas (executors)": "sum(metrics_executor_activeTasks)",
}
fig, ejes = plt.subplots(len(consultas), 1, figsize=(9, 2.2 * len(consultas)), sharex=True)
for ax, (titulo, expr) in zip(ejes, consultas.items()):
    s = m.serie(expr, minutos=15)
    if not s.empty:
        s.plot(ax=ax, legend=False)
    ax.set_title(titulo, fontsize=10); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("/workspace/entregas/F4/carga_metricas.png", dpi=120); plt.show()

**✏️ TAREA B1.** Los nombres de las métricas de Spark dependen de la versión: comprobadlos en
http://localhost:8080/metrics/master/prometheus y en http://localhost:4040/metrics/executors/prometheus.
Escribid y explicad 5 consultas PromQL útiles (usad al menos `rate`, `sum by`, `avg_over_time` y una operación
entre dos métricas). Aplicad los métodos **USE** (utilización, saturación, errores) a los recursos y **RED**
(peticiones, errores, duración) al servicio.

## C · Cuadro de mando
En Grafana (carpeta *BDA Puerto*) hay un cuadro de mando inicial. **✏️ TAREA C1.** Completadlo siguiendo el panel
«TAREA» y exportad el JSON (*Share → Export → Save to file*) a `docker/grafana/dashboards/salud_cluster_final.json`.
Debe responder de un vistazo: ¿está sano HDFS?, ¿tiene capacidad Spark?, ¿llegan datos?, ¿hay alertas?

## D · Alertas
Las reglas están en `docker/prometheus/alertas.yml` y se ven en http://localhost:9090/alerts.
Tras editar el fichero, recargad Prometheus con la celda siguiente.

**✏️ TAREA D1.** Añadid **al menos 3 alertas propias** (severidad, umbral y `for` justificados). Probad cada alerta
provocando el fallo y midiendo el **tiempo de detección** (MTTD) con `esperar_alerta`:

| Alerta | Cómo provocarla (PowerShell) |
|---|---|
| DataNodeMuerto | `docker stop bda-puerto-datanode-3` (y luego `docker start ...`) |
| BloquesCorruptos | procedimiento de corrupción de la fase 3 |
| WorkersSparkInsuficientes | `docker stop bda-puerto-spark-worker-2` |
| DatosObsoletos (vuestra) | lanzad y luego parad `scripts/hdfs/simular_llegada_ais.sh` |

In [ ]:
print(requests.post("http://prometheus:9090/-/reload", timeout=10).status_code, "(200 = recargado)")
pd.DataFrame(m.alertas())

In [ ]:
def esperar_alerta(nombre, limite_s=600):
    '''Ejecutad esta función JUSTO después de provocar el fallo. Devuelve los segundos hasta «firing».'''
    t0 = time.time()
    while time.time() - t0 < limite_s:
        estados = {a["alerta"]: a["estado"] for a in m.alertas()}
        if estados.get(nombre) == "firing":
            seg = round(time.time() - t0)
            print(f"{nombre} disparada tras {seg} s")
            return seg
        time.sleep(5)
    print("No se disparó en", limite_s, "s"); return None

# Ejemplo:  mttd_datanode = esperar_alerta("DataNodeMuerto")

**✏️ TAREA D2.** Configurad en Grafana un **punto de contacto** (*Alerting → Contact points*; p. ej. un correo de
prueba o un *webhook*) y explicad cómo evitaríais la **fatiga de alertas** (agrupación, silencios, severidades).

## E · ¿Es eficiente la monitorización?
Monitorizar también consume recursos. Medimos cuánto y cuánto tardan las consultas.

In [ ]:
def medir(expr, rango=False, n=10):
    tiempos = []
    for _ in range(n):
        t0 = time.perf_counter()
        (m.serie(expr, minutos=60) if rango else m.consulta(expr))
        tiempos.append(time.perf_counter() - t0)
    return round(1000 * sorted(tiempos)[len(tiempos) // 2], 1)

rend = pd.DataFrame([
    {"consulta": "instantánea simple", "ms_mediana": medir("bda_hdfs_datanodes_vivos")},
    {"consulta": "rango 1 h con rate", "ms_mediana": medir('rate(node_cpu_seconds_total[1m])', rango=True)},
    {"consulta": "agregación global", "ms_mediana": medir('sum by (job) (up)')},
])
extra = {
    "series activas en Prometheus": m.valor("prometheus_tsdb_head_series"),
    "memoria de Prometheus (MB)": round((m.valor('process_resident_memory_bytes{job="prometheus"}') or 0) / 1e6, 1),
    "duración media del scrape (ms)": round(1000 * (m.valor("avg(scrape_duration_seconds)") or 0), 1),
    "respuesta del NameNode al exportador (ms)": round(1000 * (m.valor("bda_namenode_respuesta_segundos") or 0), 1),
}
print(extra)
rend

**✏️ TAREA E1.** Cambiad `scrape_interval` a `5s` en `docker/prometheus/prometheus.yml`, recargad, dejad 15 minutos de
carga y repetid la medición. Tabla comparativa: series, memoria, duración del scrape, resolución obtenida.
¿Qué intervalo recomendáis y por qué? ¿Qué es la **cardinalidad** y por qué hay que vigilarla?

## F · Fiabilidad: ¿dicen la verdad las métricas?
Comparamos lo que dicen las métricas con la **verdad** obtenida por otra vía (fsck, WebHDFS, API del master).

In [ ]:
informe_fsck = h.fsck("/", bloques=False, ubicaciones=False)
def dato_fsck(patron):
    r = re.search(patron, informe_fsck)
    return int(r.group(1)) if r else None

master = requests.get("http://spark-master:8080/json/", timeout=10).json()
res = h.resumen("/")
contraste = pd.DataFrame([
    ("DataNodes vivos", m.valor("bda_hdfs_datanodes_vivos"), len([d for d in h.datanodes() if d["estado"] == "vivo"])),
    ("Bloques totales", m.valor("bda_hdfs_bloques_total"), dato_fsck(r"Total blocks \(validated\):\s+(\d+)")),
    ("Bloques corruptos", m.valor("bda_hdfs_bloques_corruptos"), dato_fsck(r"Corrupt blocks:\s+(\d+)")),
    ("Ficheros + carpetas", m.valor("bda_hdfs_ficheros_total"), res["ficheros"] + res["carpetas"]),
    ("Workers Spark vivos", m.valor("metrics_master_aliveWorkers_Value"), master.get("aliveworkers")),
], columns=["indicador", "según Prometheus", "verdad (otra fuente)"])
contraste["coincide"] = contraste["según Prometheus"] == contraste["verdad (otra fuente)"]
contraste.to_csv("/workspace/entregas/F4/fiabilidad.csv", index=False)
contraste

**✏️ TAREA F1.** Explicad cualquier discrepancia (¿retraso del *scrape*?, ¿definiciones distintas?). Parad el
exportador (`docker compose --profile monitor stop hdfs-exporter`) 3 minutos: ¿qué muestra Grafana?, ¿se dispara
alguna alerta?, ¿qué significa que una serie quede «obsoleta» (*stale*)? ¿Cómo sabemos que el propio sistema de
monitorización funciona? (pista: *watchdog* / alerta que siempre está activa).

## G · Prueba de estabilidad con caos controlado
Durante **30-60 minutos** ejecutamos la consulta tipo cada 20 s y registramos latencia y errores, mientras
`scripts/caos.ps1` para y arranca contenedores al azar **en otra ventana de PowerShell**:
```powershell
.\scripts\caos.ps1 -Minutos 30 -PausaMin 60 -PausaMax 180
```

In [ ]:
MINUTOS = 30
registro = []
fin = time.time() + MINUTOS * 60
while time.time() < fin:
    t0 = time.perf_counter(); ok = True; error = ""
    try:
        consulta_tipo()
    except Exception as e:
        ok, error = False, str(e)[:120]
    registro.append({"ts": pd.Timestamp.now(), "ok": ok, "latencia_s": round(time.perf_counter() - t0, 2), "error": error})
    time.sleep(20)
est = pd.DataFrame(registro)
est.to_csv("/workspace/entregas/F4/estabilidad.csv", index=False)
print("Disponibilidad: %.2f %%" % (100 * est.ok.mean()))
print(est.latencia_s.describe(percentiles=[.5, .95, .99]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(est.ts, est.latencia_s, ".-", label="latencia consulta tipo (s)")
ax.scatter(est[~est.ok].ts, [est.latencia_s.max()] * (~est.ok).sum(), color="red", marker="x", label="error")
ax.axhline(est.latencia_s.quantile(.95), ls="--", color="grey", label="p95")
ax.legend(); ax.grid(alpha=.3); ax.set_title("Estabilidad del servicio durante la prueba de caos")
plt.tight_layout(); plt.savefig("/workspace/entregas/F4/estabilidad.png", dpi=120); plt.show()

**✏️ TAREA G1.** Cruzad `estabilidad.csv` con `entregas/F4/caos_registro.csv` y con las alertas: para cada fallo
provocado, ¿se detectó (MTTD)?, ¿cuánto tardó en recuperarse (MTTR)?, ¿afectó a los usuarios (errores o latencia)?
Proponed un **SLO** razonable (p. ej. «el 99 % de las consultas tipo responden en menos de X s») y decid si se cumple.
Usad también PromQL: `avg_over_time(up[1h])` y `changes(bda_hdfs_datanodes_vivos[1h])`.

In [ ]:
spark.stop()